In [3]:
using SymPy
using PyCall
using DelimitedFiles

cd("c:/FD")
pushfirst!(PyCall.PyVector(PyCall.pyimport("sys")."path"), pwd())

# 导入模块
importlib = pyimport("importlib")
fH_func = pyimport("fH_func")
importlib.reload(fH_func)
fH_class = pyimport("fH_class")
importlib.reload(fH_class)
grid_data = pyimport("grid_data")
importlib.reload(grid_data)
func_terms = fH_class.func_terms_class(n_max=5)
#func_terms = fH_func
# 获取参数
kx = convert(Float64, grid_data."kx_val")
ky = convert(Float64, grid_data."ky_val")
kz = convert(Float64, grid_data."kz_val")
b  = convert(Float64, grid_data."b_val")
h  = convert(Float64, grid_data."h_val")
mu = convert(Float64, grid_data."mu_val")
a0 = convert(Float64, grid_data."a0_val")
b0 = convert(Float64, grid_data."b0_val")
x_full = convert(Array{Float64}, grid_data."x")
y_full = convert(Array{Float64}, grid_data."y")
z_full = convert(Array{Float64}, grid_data."z")

sympy = pyimport("sympy")

# 构建完整表达式
println("构建符号表达式...")
total_expr_cos = sympy.sympify(0)
total_expr_sin = sympy.sympify(0)

for i in 1:length(func_terms.terms1)
    exp_part, poly_part = func_terms.terms1[i]
    term = exp_part * poly_part
    total_expr_cos = total_expr_cos + term
end
total_expr_cos = total_expr_cos * func_terms.term_cos

for i in 1:length(func_terms.terms2)
    exp_part, poly_part = func_terms.terms2[i]
    term = exp_part * poly_part
    total_expr_sin = total_expr_sin + term
end
total_expr_sin = total_expr_sin * func_terms.term_sin

total_expr = total_expr_cos + total_expr_sin

julia_code = sympy.julia_code(total_expr)
expr = Meta.parse(julia_code)

using MacroTools

function devectorize(expr)
    MacroTools.postwalk(expr) do x
        if @capture(x, a_.b_)
            return :($a.$b)
        elseif x isa Expr && x.head == :.
            return Expr(:call, x.args[1], x.args[2:end]...)
        elseif x isa Expr && x.head == :call
            if x.args[1] isa Symbol
                op_str = string(x.args[1])
                if startswith(op_str, ".")
                    new_op = Symbol(op_str[2:end])
                    return Expr(:call, new_op, x.args[2:end]...)
                end
            end
        end
        return x
    end
end

function build_and_save_f1(julia_code::AbstractString, filename::String)
    expr = Meta.parse(julia_code)
    devec_expr = devectorize(expr)
    
    # 保存函数到文件
    func_code = """
    function f1(x::Float64, y::Float64, z::Float64,
                kx::Float64, ky::Float64, kz::Float64,
                b::Float64, h::Float64, mu::Float64)
        return $(devec_expr)
    end
    """
    
    open(filename, "w") do io
        write(io, func_code)
    end
    
    println("函数已保存到: $filename")
    
    # 返回匿名函数
    f_1 = eval(quote
        (x::Float64, y::Float64, z::Float64,
         kx::Float64, ky::Float64, kz::Float64,
         b::Float64, h::Float64, mu::Float64) -> $(devec_expr)
    end)
    
    return f_1
end

# 性能测试函数 - 注意这里参数化了格点数
function runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, n_points, n_iterations)
    result = similar(x)
    for j in 1:n_iterations
        @inbounds for i in 1:n_points
            result[i] = f2(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
        end
    end
    return result
end

# 生成函数
println("生成Julia函数...")
f_test = build_and_save_f1(julia_code, "generated_f.jl")

构建符号表达式...
生成Julia函数...
函数已保存到: generated_f.jl


#24 (generic function with 1 method)

In [7]:
result = runloop!(f_test, x_full, y_full, z_full, kx, ky, kz, b, h, mu, n_points, n_iterations)

UndefVarError: UndefVarError: `n_points` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [8]:
# ========== 测试不同格点数量 ==========
grid_sizes = collect(8000:1000:8000)
times = Float64[]
n_iterations = 1

println("\n开始性能测试（不保存exp版本）...")
println("=" ^ 60)

for n_grid in grid_sizes
    println("\n测试格点数: $n_grid")
    
    # 截取数据
    x = x_full[1:n_grid]
    y = y_full[1:n_grid]
    z = z_full[1:n_grid]
    
    result = zeros(Float64, n_grid)
    
    # 预热
    println("  预热中...")
    runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, n_grid, 10)
    result = runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, n_grid, n_iterations)
    # 正式计时
    println("  正式计时...")
    elapsed_time = @elapsed runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, n_grid, n_iterations)
    
    avg_time = elapsed_time / n_iterations * 1000  # 转换为毫秒
    push!(times, elapsed_time)
    
    println("  平均单次耗时: $(round(avg_time, digits=3)) ms")
    println("  总耗时: $(round(elapsed_time, digits=3)) s")
end

println("\n" * "=" ^ 60)
println("测试完成！")

# ========== 保存结果到CSV ==========
data_matrix = hcat(grid_sizes, times)
writedlm("benchmark_no_exp_storage.csv", 
         vcat(["grid_size" "time_ms"], data_matrix), 
         ',')

println("\n结果已保存到: benchmark_no_exp_storage.csv")
println("\n格点数  |  耗时(s)")
println("-" ^ 30)
for (n, t) in zip(grid_sizes, times)
    println("$n  |  $(round(t, digits=3))")
end


开始性能测试（不保存exp版本）...

测试格点数: 8000
  预热中...
  正式计时...
  平均单次耗时: 2.112 ms
  总耗时: 0.002 s

测试完成！

结果已保存到: benchmark_no_exp_storage.csv

格点数  |  耗时(s)
------------------------------
8000  |  0.002


In [9]:
result

8000-element Vector{Float64}:
 -0.3017228023624591
 -0.30496787471424097
 -0.30926081310286485
 -0.3146033334973271
 -0.3208273493866808
 -0.32756242391610296
 -0.33424379615710936
 -0.34017393611472085
 -0.3446357333505626
 -0.3470353050402086
  ⋮
 -0.3446357333505626
 -0.34017393611472085
 -0.33424379615710936
 -0.32756242391610296
 -0.3208273493866808
 -0.3146033334973271
 -0.30926081310286485
 -0.30496787471424097
 -0.3017228023624591

In [2]:
f_test = build_and_save_f1(julia_code, "generated_f.jl")
@time runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, 300)
@time runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, 300)

函数已保存到: generated_f.jl
  5.795315 seconds (1.57 M allocations: 69.194 MiB, 5.03% compilation time)
  5.689778 seconds (3 allocations: 62.585 KiB)


8000-element Vector{Float64}:
 -0.3017228023624591
 -0.30496787471424097
 -0.30926081310286485
 -0.3146033334973271
 -0.3208273493866808
 -0.32756242391610296
 -0.33424379615710936
 -0.34017393611472085
 -0.3446357333505626
 -0.3470353050402086
  ⋮
 -0.3446357333505626
 -0.34017393611472085
 -0.33424379615710936
 -0.32756242391610296
 -0.3208273493866808
 -0.3146033334973271
 -0.30926081310286485
 -0.30496787471424097
 -0.3017228023624591

In [4]:
@time runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, 3000)

  8.976046 seconds (3 allocations: 62.601 KiB)


8000-element Vector{Float64}:
 0.2989926480692031
 0.30086253720864103
 0.3033576523225069
 0.3064846267909936
 0.31014729708595795
 0.3141262915483655
 0.3180839799339403
 0.3216023537758981
 0.3242518871102745
 0.3256773966056631
 ⋮
 0.3242518871102745
 0.3216023537758981
 0.3180839799339403
 0.3141262915483655
 0.31014729708595795
 0.3064846267909936
 0.3033576523225069
 0.30086253720864103
 0.2989926480692031

In [7]:
@time runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, 3000)

  8.762613 seconds (3 allocations: 62.616 KiB)


8000-element Vector{Float64}:
 0.2989926480692031
 0.30086253720864103
 0.3033576523225069
 0.3064846267909936
 0.31014729708595795
 0.3141262915483655
 0.3180839799339403
 0.3216023537758981
 0.3242518871102745
 0.3256773966056631
 ⋮
 0.3242518871102745
 0.3216023537758981
 0.3180839799339403
 0.3141262915483655
 0.31014729708595795
 0.3064846267909936
 0.3033576523225069
 0.30086253720864103
 0.2989926480692031

In [3]:
include("generated_f1.jl")

# 现在可以直接使用 f2 函数
result = f1(1.0, 2.0, 3.0, 1.5, 2.0, 3.0, 0.5, 0.1, 0.2)

-0.051962456291930546

In [14]:
@time runloop!(f_test, x, y, z, kx, ky, kz, b, h, mu, 300)

  0.674507 seconds (3 allocations: 62.616 KiB)


8000-element Vector{Float64}:
 0.2989926480692031
 0.30086253720864103
 0.3033576523225069
 0.3064846267909936
 0.31014729708595795
 0.3141262915483655
 0.3180839799339403
 0.3216023537758981
 0.3242518871102745
 0.3256773966056631
 ⋮
 0.3242518871102745
 0.3216023537758981
 0.3180839799339403
 0.3141262915483655
 0.31014729708595795
 0.3064846267909936
 0.3033576523225069
 0.30086253720864103
 0.2989926480692031

In [ ]:
f = build_and_save_f1(julia_code, "generated_f.jl")

函数已保存到: generated_f1.jl


#39 (generic function with 1 method)

In [61]:
f_2(1.0, 2.0, 3.0, 1.5, 2.0, 3.0, 0.5, 0.1, 0.2)

-0.051962456291930546

In [ ]:
func_terms.func_terms.terms1

4-element Vector{Tuple{PyObject, PyObject}}:
 (PyObject exp(-0.1*x**2 - 0.1*y**2 - 0.1*z**2), PyObject -0.0002*kx**7*x - 0.0002*kx**6*ky*y - 0.0002*kx**6*kz*z - 0.0006*kx**5*ky**2*x - 0.0006*kx**5*kz**2*x + 3.2e-5*kx**5*x**3 + 1.6e-5*kx**5*x*y**2 + 1.6e-5*kx**5*x*z**2 + 0.01136*kx**5*x - 0.0006*kx**4*ky**3*y - 0.0006*kx**4*ky**2*kz*z - 0.0006*kx**4*ky*kz**2*y + 6.4e-5*kx**4*ky*x**2*y + 1.6e-5*kx**4*ky*y**3 + 1.6e-5*kx**4*ky*y*z**2 + 0.01136*kx**4*ky*y - 0.0006*kx**4*kz**3*z + 6.4e-5*kx**4*kz*x**2*z + 1.6e-5*kx**4*kz*y**2*z + 1.6e-5*kx**4*kz*z**3 + 0.01136*kx**4*kz*z - 0.0006*kx**3*ky**4*x - 0.0012*kx**3*ky**2*kz**2*x + 4.8e-5*kx**3*ky**2*x**3 + 8.0e-5*kx**3*ky**2*x*y**2 + 3.2e-5*kx**3*ky**2*x*z**2 + 0.02272*kx**3*ky**2*x + 9.6e-5*kx**3*ky*kz*x*y*z - 0.0006*kx**3*kz**4*x + 4.8e-5*kx**3*kz**2*x**3 + 3.2e-5*kx**3*kz**2*x*y**2 + 8.0e-5*kx**3*kz**2*x*z**2 + 0.02272*kx**3*kz**2*x - 9.92000000000001e-7*kx**3*x**5 - 1.472e-6*kx**3*x**3*y**2 - 1.472e-6*kx**3*x**3*z**2 - 0.00089568*kx**3*x**3 - 

:(((((-0.12kx) .* x - (0.12ky) .* y) - (0.12kz) .* z) .* exp((-0.3 * x .^ 2 - 0.3 * y .^ 2) - 0.3 * z .^ 2) + ((((((((((((((((0.024 * kx .^ 3) .* x + ((0.024 * kx .^ 2) .* ky) .* y + ((0.024 * kx .^ 2) .* kz) .* z + ((0.024kx) .* ky .^ 2) .* x + ((0.024kx) .* kz .^ 2) .* x) - (0.00192kx) .* x .^ 3) - ((0.00192kx) .* x) .* y .^ 2) - ((0.00192kx) .* x) .* z .^ 2) - (0.4528kx) .* x) + (0.024 * ky .^ 3) .* y + ((0.024 * ky .^ 2) .* kz) .* z + ((0.024ky) .* kz .^ 2) .* y) - ((0.00192ky) .* x .^ 2) .* y) - (0.00192ky) .* y .^ 3) - ((0.00192ky) .* y) .* z .^ 2) - (0.4528ky) .* y) + (0.024 * kz .^ 3) .* z) - ((0.00192kz) .* x .^ 2) .* z) - ((0.00192kz) .* y .^ 2) .* z) - (0.00192kz) .* z .^ 3) - (0.4528kz) .* z) .* exp((-0.2 * x .^ 2 - 0.2 * y .^ 2) - 0.2 * z .^ 2) + (((((((((((((((((((((((((((((((((((((((((((((((((((((((((-0.0012 * kx .^ 5) .* x - ((0.0012 * kx .^ 4) .* ky) .* y) - ((0.0012 * kx .^ 4) .* kz) .* z) - ((0.0024 * kx .^ 3) .* ky .^ 2) .* x) - ((0.0024 * kx .^ 3) .* kz .^ 2) .* x)

In [55]:
devec_expr = devectorize(expr)
println("去向量化后: ", devec_expr)

# 生成函数
eval(quote
    function f1(x, y, z, kx, ky, kz, b, h, mu)
        return $(devec_expr)
    end
end)

去向量化后: ((((-0.12kx) * x - (0.12ky) * y) - (0.12kz) * z) * exp((-0.3 * x ^ 2 - 0.3 * y ^ 2) - 0.3 * z ^ 2) + ((((((((((((((((0.024 * kx ^ 3) * x + ((0.024 * kx ^ 2) * ky) * y + ((0.024 * kx ^ 2) * kz) * z + ((0.024kx) * ky ^ 2) * x + ((0.024kx) * kz ^ 2) * x) - (0.00192kx) * x ^ 3) - ((0.00192kx) * x) * y ^ 2) - ((0.00192kx) * x) * z ^ 2) - (0.4528kx) * x) + (0.024 * ky ^ 3) * y + ((0.024 * ky ^ 2) * kz) * z + ((0.024ky) * kz ^ 2) * y) - ((0.00192ky) * x ^ 2) * y) - (0.00192ky) * y ^ 3) - ((0.00192ky) * y) * z ^ 2) - (0.4528ky) * y) + (0.024 * kz ^ 3) * z) - ((0.00192kz) * x ^ 2) * z) - ((0.00192kz) * y ^ 2) * z) - (0.00192kz) * z ^ 3) - (0.4528kz) * z) * exp((-0.2 * x ^ 2 - 0.2 * y ^ 2) - 0.2 * z ^ 2) + (((((((((((((((((((((((((((((((((((((((((((((((((((((((((-0.0012 * kx ^ 5) * x - ((0.0012 * kx ^ 4) * ky) * y) - ((0.0012 * kx ^ 4) * kz) * z) - ((0.0024 * kx ^ 3) * ky ^ 2) * x) - ((0.0024 * kx ^ 3) * kz ^ 2) * x) + (9.6e-5 * kx ^ 3) * x ^ 3 + ((6.4e-5 * kx ^ 3) * x) * y ^ 2 + ((6.4e-5

f1 (generic function with 1 method)

In [1]:
f1(1.0, 2.0, 3.0, 1.5, 2.0, 3.0, 0.5, 0.1, 0.2)

UndefVarError: UndefVarError: `f1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [66]:
typeof(f1)

typeof(f1) (singleton type of function f1, subtype of Function)

In [65]:
function runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, n_iterations)
    result = similar(x)
    for j in 1:n_iterations
        @inbounds for i in 1:8000
            result[i] = f2(x[i], y[i], z[i])
        end
    end
    return result
end

@time runloop!(f1, x, y, z, kx, ky, kz, b, h, mu, 30)

MethodError: MethodError: no method matching f1(::Float64, ::Float64, ::Float64)
The function `f1` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  f1(::Any, ::Any, ::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any)
   @ Main c:\FD\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y133sZmlsZQ==.jl:6


In [40]:
function count_by_type(expr, op_symbol)
    """统计特定操作符的数量"""
    if typeof(expr) != Expr
        return 0
    end
    
    count = 0
    if expr.head == :call && length(expr.args) > 0 && expr.args[1] == op_symbol
        count = 1
    end
    
    for arg in expr.args
        count += count_by_type(arg, op_symbol)
    end
    return count
end

println("加法数量: ", count_by_type(expr, :+))
println("乘法数量: ", count_by_type(expr, :*))
println("指数数量: ", count_by_type(expr, :exp))

加法数量: 45
乘法数量: 305
指数数量: 7


In [35]:
f0 = eval(:( (x, y, z, kx, ky, kz, b, h, mu) -> $expr ))
result = f0(1.0, 2.0, 3.0, 1.5, 2.0, 3.0, 0.5, 0.1, 0.2)
println(result)

-0.051962456291930546


In [ ]:
using PyCall
using SymPy

sympy = pyimport("sympy")
julia_code = sympy.julia_code(func_terms.func_terms.psi_fH)

expr = Meta.parse(julia_code)

f = eval(:( (x, y, z, kx, ky, kz, b, h, mu) -> $expr ))
result = f(1.0, 2.0, 3.0, 1.5, 2.0, 3.0, 0.5, 0.1, 0.2)
println(result)

-0.05196245629192997


In [ ]:
sympy.julia_code(func_terms.func_terms.psi_fH)

In [36]:
length(julia_code)

9078

函数已保存到: generated_f1.jl


#26 (generic function with 1 method)

找到 3 个项
函数已保存到: generated_f_reorganized.jl


#54 (generic function with 1 method)

-0.05196245629192997

In [63]:
f_2(1.0, 2.0, 3.0, 1.5, 2.0, 3.0, 0.5, 0.1, 0.2)

-0.051962456291930546

In [28]:
x.size

(8000,)

build_f1 (generic function with 1 method)

In [10]:
julia_code

"(0.0008 * (-150.0 * kx .* x - 150.0 * ky .* y - 150.0 * kz .* z) .* exp(-0.3 * x .^ 2) .* exp(-0.3 * y .^ 2) .* exp(-0.3 * z .^ 2) + 0.4 * (-1.0 * kx .* x - 1.0 * ky .* y - 1.0 * kz .* z) .* exp(-0.1 * x .^ 2) .* exp(-0.1 * y .^ 2) .* exp(-0.1 * z .^ 2) - 0.032 * (15.0 " ⋯ 8678 bytes ⋯ " exp(-0.3 * y .^ 2) .* exp(-0.3 * z .^ 2) + 10.0 * exp(-0.2 * x .^ 2) .* exp(-0.2 * y .^ 2) .* exp(-0.2 * z .^ 2) + 8.0 * exp(-0.1 * x .^ 2) .* exp(-0.1 * y .^ 2) .* exp(-0.1 * z .^ 2)) .* sin(b + kx .* x + ky .* y + kz .* z) + 1.0 * sin(b + kx .* x + ky .* y + kz .* z)"

MethodError: MethodError: no method matching (::var"#39#40")(::Float64, ::Float64, ::Float64)
The function `#39` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  (::var"#39#40")(::Float64, ::Float64, ::Float64, !Matched::Float64, !Matched::Float64, !Matched::Float64, !Matched::Float64, !Matched::Float64, !Matched::Float64)
   @ Main c:\FD\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W0sZmlsZQ==.jl:75


In [9]:
@time runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, 3000)

 11.964008 seconds (3 allocations: 62.569 KiB)


8000-element Vector{Float64}:
 0.2989926480692031
 0.30086253720864103
 0.3033576523225069
 0.3064846267909936
 0.310147297085958
 0.3141262915483656
 0.3180839799339403
 0.3216023537758981
 0.3242518871102745
 0.325677396605663
 ⋮
 0.3242518871102745
 0.3216023537758981
 0.3180839799339403
 0.3141262915483656
 0.310147297085958
 0.3064846267909936
 0.3033576523225069
 0.30086253720864103
 0.2989926480692031

  0.086700 seconds (9.39 k allocations: 570.414 KiB, 11.34% compilation time)
  8.351309 seconds (3 allocations: 62.616 KiB)


8000-element Vector{Float64}:
 0.2989926480692031
 0.30086253720864103
 0.3033576523225069
 0.3064846267909936
 0.31014729708595795
 0.3141262915483655
 0.3180839799339403
 0.3216023537758981
 0.3242518871102745
 0.3256773966056631
 ⋮
 0.3242518871102745
 0.3216023537758981
 0.3180839799339403
 0.3141262915483655
 0.31014729708595795
 0.3064846267909936
 0.3033576523225069
 0.30086253720864103
 0.2989926480692031

In [ ]:
# 看看 f1 的 LLVM 代码
@code_llvm debuginfo=:none f1(1.0, 2.0, 3.0, 1.5, 2.0, 3.0, 0.5, 0.1, 0.2)

# 如果 f1 很简单,编译器应该会自动内联
# 如果 f1 很复杂,考虑拆分或简化

In [ ]:
@code_llvm debuginfo=:none runloop!(f2, x, y, z, 1.5, 2.0, 3.0, b, h, mu, 10)


In [ ]:
# 查看类型推断后的中间表示
@code_warntype runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, 30)

In [ ]:
function runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, n_iterations)
    result = similar(x)
    N = length(x)
    
    for j in 1:n_iterations
        @inbounds @simd for i in 1:N  # 添加 @simd 启用向量化
            result[i] = f2(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
        end
    end
    return result
end

@time runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, 1)
@time runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, 300)


In [ ]:
@time runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, 300)

In [ ]:
function runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, n_iterations)
    result = similar(x)
    for j in 1:n_iterations
        @inbounds for i in 1:8000
            result[i] = f2(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
        end
    end
    return result
end

@time runloop!(f2, x, y, z, kx, ky, kz, b, h, mu, 3000)


In [ ]:
n_iterations = 30
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:8000
            result[i] = f2(x[i], y[i], z[i], kx, ky, kz, b, h, mu)          
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
n_iterations = 20
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f2($x[i], $y[i], $z[i], $kx, $ky, $kz, $b, $h, $mu)
            
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
length(x)

In [ ]:
n_iterations = 40
g(x,y) = x+y
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            res = g(0,1)
            
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
x = convert(Array{Float64}, grid_data."x")
y = convert(Array{Float64}, grid_data."y")
n_iterations = 80
len_x = length(x)
g(x,y) = x+y

println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:len_x
            res = g(x[i],y[i])
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
function run_simulation(x::Vector{Float64}, y::Vector{Float64}, n_iterations::Int)
    len_x = length(x)
    g(x, y) = x + y
    for j in 1:n_iterations
        for i in 1:len_x
            res = g(x[i], y[i])
        end
    end
    return nothing
end

x = convert(Vector{Float64}, grid_data."x")
y = convert(Vector{Float64}, grid_data."y")

println("开始...")
@time begin
    for j in 1:40
        run_simulation(x, y, 80)
    end
end

println("完成!")


In [ ]:
using Pkg
Pkg.add("BenchmarkTools")


In [ ]:
using BenchmarkTools

@btime run_simulation($x, $y, 80)


In [ ]:
n_iterations = 1000
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f1(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
            
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
n_iterations = 10000
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f1(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
            
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
n_iterations = 100000
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f1(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
            
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
n_iterations = 1000000
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f1(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
            
        end
    end
end
GC.gc()
println("完成!")

In [ ]:
n_iterations = 60
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f1(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
        end
    end
end
println("完成!")

In [ ]:
using Printf

# 测试原地修改数组（使用 .= 或者 ! 函数）
function test_inplace_modification(n_iterations::Int, array_size::Int)
    println("\n========== 原地修改测试 ==========")
    
    # 创建数组
    arr = zeros(Float64, array_size)
    
    # 记录初始内存
    GC.gc()  # 先进行垃圾回收
    initial_memory = Base.gc_live_bytes() / 1024^2  # 转换为 MB
    
    println("初始内存: $(round(initial_memory, digits=2)) MB")
    println("数组大小: $array_size 个元素")
    println("循环次数: $n_iterations")
    
    # 进行原地修改
    for i in 1:n_iterations
        arr .= arr .+ 1.0  # 原地修改
        
        if i % 1000 == 0
            current_memory = Base.gc_live_bytes() / 1024^2
            @printf("迭代 %d: 当前内存 %.2f MB\n", i, current_memory)
        end
    end
    
    # 记录最终内存
    GC.gc()
    final_memory = Base.gc_live_bytes() / 1024^2
    
    println("最终内存: $(round(final_memory, digits=2)) MB")
    println("内存增长: $(round(final_memory - initial_memory, digits=2)) MB")
    
    return final_memory - initial_memory
end

# 测试非原地修改数组（创建新数组）
function test_non_inplace_modification(n_iterations::Int, array_size::Int)
    println("\n========== 非原地修改测试 ==========")
    
    # 创建数组
    arr = zeros(Float64, array_size)
    
    # 记录初始内存
    GC.gc()  # 先进行垃圾回收
    initial_memory = Base.gc_live_bytes() / 1024^2
    
    println("初始内存: $(round(initial_memory, digits=2)) MB")
    println("数组大小: $array_size 个元素")
    println("循环次数: $n_iterations")
    
    # 进行非原地修改
    for i in 1:n_iterations
        arr = arr .+ 1.0  # 创建新数组
        
        if i % 1000 == 0
            current_memory = Base.gc_live_bytes() / 1024^2
            @printf("迭代 %d: 当前内存 %.2f MB\n", i, current_memory)
        end
    end
    
    # 记录最终内存
    GC.gc()
    final_memory = Base.gc_live_bytes() / 1024^2
    
    println("最终内存: $(round(final_memory, digits=2)) MB")
    println("内存增长: $(round(final_memory - initial_memory, digits=2)) MB")
    
    return final_memory - initial_memory
end

# 更详细的内存分配测试
function test_memory_allocation(n_iterations::Int, array_size::Int)
    println("\n========== 内存分配统计测试 ==========")
    println("数组大小: $array_size 个元素")
    println("循环次数: $n_iterations")
    
    # 原地修改的内存分配
    println("\n--- 原地修改 ---")
    arr1 = zeros(Float64, array_size)
    alloc1 = @allocated begin
        for i in 1:n_iterations
            arr1 .= arr1 .+ 1.0
        end
    end
    println("总分配内存: $(round(alloc1 / 1024^2, digits=2)) MB")
    println("平均每次迭代: $(round(alloc1 / n_iterations / 1024, digits=2)) KB")
    
    # 非原地修改的内存分配
    println("\n--- 非原地修改 ---")
    arr2 = zeros(Float64, array_size)
    alloc2 = @allocated begin
        for i in 1:n_iterations
            arr2 = arr2 .+ 1.0
        end
    end
    println("总分配内存: $(round(alloc2 / 1024^2, digits=2)) MB")
    println("平均每次迭代: $(round(alloc2 / n_iterations / 1024, digits=2)) KB")
    
    println("\n--- 对比 ---")
    println("非原地修改分配的内存是原地修改的: $(round(alloc2 / alloc1, digits=2)) 倍")
end

# 主测试函数
function main()
    println("Julia 内存测试：原地修改 vs 非原地修改")
    println("=" ^ 50)
    
    # 测试参数
    array_size = 1_000_000  # 100万个元素
    n_iterations = 5000      # 5000次迭代
    
    # 运行测试
    inplace_growth = test_inplace_modification(n_iterations, array_size)
    
    # 等待一下，让系统稳定
    sleep(1)
    
    non_inplace_growth = test_non_inplace_modification(n_iterations, array_size)
    
    # 详细的内存分配测试
    test_memory_allocation(n_iterations, array_size)
    
    # 总结
    println("\n" * "=" ^ 50)
    println("总结:")
    println("原地修改内存增长: $(round(inplace_growth, digits=2)) MB")
    println("非原地修改内存增长: $(round(non_inplace_growth, digits=2)) MB")
    println("\n结论:")
    println("- 原地修改: 内存使用与循环次数基本无关（仅有少量临时分配）")
    println("- 非原地修改: 每次循环都会分配新内存，总内存与循环次数成正比")
end

# 运行主函数
main()

In [ ]:
psi_expr = sympify(func_terms.func_terms.psi_fH)

# 定义符号变量
@syms x y z kx ky kz b h mu

# 现在可以 lambdify
f = lambdify(psi_expr, (x, y, z, kx, ky, kz, b, h, mu))

In [ ]:
@code_warntype f(x[1], y[1], z[1], kx, ky, kz, b, h, mu)

In [ ]:
n_iterations = 20
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
        end
    end
end
println("完成!")

In [ ]:
string(func_terms.func_terms.psi_fH)

In [ ]:
n_iterations = 40
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
        end
    end
end
println("完成!")

In [ ]:
expr = Meta.parse(julia_code)

# 用宏生成真正的函数定义
eval(quote
    function f1(x, y, z, kx, ky, kz, b, h, mu)
        return $(expr)
    end
end)

In [ ]:
n_iterations = 40
result = similar(x)
println("开始 $n_iterations 次迭代...")
@time begin
    for j in 1:n_iterations
        for i in 1:length(x)
            result[i] = f1(x[i], y[i], z[i], kx, ky, kz, b, h, mu)
        end
    end
end
println("完成!")

In [ ]:
using BenchmarkTools
@btime f1($x[1], $y[1], $z[1], $kx, $ky, $kz, $b, $h, $mu)


In [ ]:
using SymPy

# 创建符号表达式
@syms x y z
expr = sin(x) + y^2 - exp(z)

# 转换为 Julia 函数（自动广播）
f = lambdify(expr, (x, y, z))

# 使用（自动支持广播）
result = f(1.0, 2.0, 3.0)           # 标量输入
result = f.([1.0, 2.0], 2.0, 3.0)  # 向量输入（手动加点）

In [ ]:
function add_broadcast_all!(jcode::String)
    # 常用数学函数
    funcs = ["exp", "sin", "cos", "tan", "sqrt", "log"]
    for f in funcs
        # 函数调用前没有点则加上
        pattern = Regex("(?<!\\.)\\b$f\\(")
        jcode = replace(jcode, pattern => "$f.(")
    end

    # 所有二元运算符加点
    ops = ["\\+", "-", "\\*", "/", "\\^"]
    for op in ops
        # 匹配前后不是点的情况，替换成带点的
        pattern = Regex("(?<!\\.)$op")
        jcode = replace(jcode, pattern => ".$op")
    end

    # 对一元负号统一加点（可选）
    # jcode = replace(jcode, r"(?<![0-9A-Za-z_])-" => ".-")

    return jcode
end


# 假设 julia_code 是 SymPy 导出的原始字符串
julia_code_broadcast = add_broadcast!(julia_code)

# 然后 eval


In [ ]:
using SymPy

# 创建符号表达式
@syms x y z
expr = sin(x) + y^2 - exp(z)

# 转换为 Julia 函数（自动广播）
f = lambdify(expr, (x, y, z))

# 使用（自动支持广播）
result = f(1.0, 2.0, 3.0)           # 标量输入
result = f.([1.0, 2.0], 2.0, 3.0)  # 向量输入（手动加点）

In [ ]:
# 预分配结果数组（如果需要保存所有结果）
n_iterations = 3000

# 计时整个循环
println("开始 $n_iterations 次迭代...")
@time begin
    for i in 1:n_iterations
        result = f.(x, y, z, kx, ky, kz, b, h, mu, a0, b0)
    end
end
println("完成!")

In [ ]:
# 假设你已经有 julia_code 字符串
# 替换 x,y,z 为 X,Y,Z
julia_code_on_grid = replace(julia_code, "x" => "X", "y" => "Y", "z" => "Z")

# 评估
psi_fH = eval(Meta.parse(julia_code_on_grid))

println(size(psi_fH))  # (N, N, N)
